# Lung CT Segmentation Demo

A short walkthrough of the pipeline: load config, inspect a sample, load a trained checkpoint, run inference, and visualize the result.

See `README.md` and `docs/technical_report.md` for full methodology and results.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

from src.config import load_config, resolve_path
from src.preprocessing import load_image_grayscale, load_mask, resize_mask, binarize_mask
from src.predict import load_model_for_inference, predict_single_image
from src.visualization import plot_sample_panel
import numpy as np
import matplotlib.pyplot as plt

cfg = load_config('../config.yaml')
cfg

In [ ]:
# Load the dataset split
with open(resolve_path('data/splits.json')) as f:
    split = json.load(f)
print('train/val/test images:', len(split['train_files']), len(split['val_files']), len(split['test_files']))

In [ ]:
# Load a trained checkpoint (run scripts/train_model.py first if this errors)
ckpt_path = resolve_path('outputs/checkpoints/unet_aug_bce_dice_best.pt')
assert os.path.exists(ckpt_path), 'Checkpoint not found -- run scripts/train_model.py first.'
model, norm_stats, device = load_model_for_inference(ckpt_path, cfg)
print('Loaded model on', device)

In [ ]:
# Run inference on one test image and visualize
images_dir = resolve_path(cfg.data.images_dir)
masks_dir = resolve_path(cfg.data.masks_dir)
fname = split['test_files'][0]

img = load_image_grayscale(os.path.join(images_dir, fname))
gt_raw = load_mask(os.path.join(masks_dir, fname))
gt_resized = resize_mask(gt_raw, cfg.data.image_size)
gt_bin = binarize_mask(gt_resized, cfg.data.mask_threshold)

prob_map, pred_bin, resized_img = predict_single_image(model, norm_stats, device, img, image_size=cfg.data.image_size)

plot_sample_panel(resized_img.astype(np.float32)/255.0, gt_bin, pred_bin, '/tmp/demo_panel.png', title=fname)
plt.figure(figsize=(18,4))
plt.imshow(plt.imread('/tmp/demo_panel.png'))
plt.axis('off')
plt.show()